In [8]:
import requests
import os
import time
from datetime import datetime, timedelta
import zipfile
import pytz

# Base URL template (with placeholder for camera ID)
base_url_template = "https://informo.madrid.es/cameras/Camara{camera_id}.jpg"

# Directory to save images in Colab
output_dir = "camera_images"
os.makedirs(output_dir, exist_ok=True)

def download_image(camera_id):
    """
    Download an image from the camera URL with a random `rand` parameter.
    """
    try:
        # Generate a random number or timestamp for the `rand` parameter
        rand_param = int(time.time() * 1000)  # Current timestamp in milliseconds

        # Construct the full URL using the camera ID
        url = base_url_template.format(camera_id=camera_id)
        url_with_rand = f"{url}?rand={rand_param}"

        # Get the current time in UTC
        current_time_utc = datetime.utcnow()
        madrid_offset = timedelta(hours=1)
        current_time_madrid = current_time_utc + madrid_offset

        # Generate a unique filename using the current timestamp
        timestamp = current_time_madrid.strftime("%Y%m%d_%H%M")
        filename = f"camera_{camera_id}_{timestamp}.jpg"
        save_path = os.path.join(output_dir, filename)

        # Download the image
        response = requests.get(url_with_rand, stream=True)
        response.raise_for_status()  # Raise an error for bad responses (4xx or 5xx)

        # Save the image
        with open(save_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
        print(f"Downloaded: {save_path}")
    except Exception as e:
        print(f"Failed to download image for camera {camera_id}: {e}")

def zip_images():
    """
    Zip the downloaded images into a single file.
    """
    zip_filename = "camera_images.zip"
    with zipfile.ZipFile(zip_filename, "w") as zipf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                zipf.write(os.path.join(root, file), file)
    print(f"Zipped images into: {zip_filename}")
    return zip_filename

def main():
    """
    Download images for specific cameras every 5 minutes.
    """
    # List of camera IDs
    #camera_ids = ["01315", "01301","01314","01335", "01318", "02302", "07303", "04313", "04314", "04315", "06314", "06313", "09301","09318", "05306", "15318"]
    camera_ids = ["01315"]

    # Number of iterations (e.g., run for 1 hour = 12 iterations)
    iterations = 5
    for i in range(iterations):
        print(f"Iteration {i + 1} at {datetime.now()}")
        for camera_id in camera_ids:
            download_image(camera_id)
            time.sleep(2)  # Wait for 2 seconds between downloads
        if (i+1<iterations):
          print("Waiting for 5 minutes...")
          time.sleep(294)  # Wait for 5 minutes (300 seconds)

        #296 -> there's a delay between iterations of almost 4 seconds.
        #If more cameras or less, adjust the value, to reduce the delay

    # Zip the images and download
    zip_filename = zip_images()
    from google.colab import files
    files.download(zip_filename)

if __name__ == "__main__":
    main()

Iteration 1 at 2025-06-08 23:31:47.935327
Downloaded: camera_images/camera_01315_20250609_0031.jpg
Waiting for 5 minutes...
Iteration 2 at 2025-06-08 23:36:44.209092
Downloaded: camera_images/camera_01315_20250609_0036.jpg
Waiting for 5 minutes...
Iteration 3 at 2025-06-08 23:41:40.461163
Downloaded: camera_images/camera_01315_20250609_0041.jpg
Waiting for 5 minutes...
Iteration 4 at 2025-06-08 23:46:36.795801
Downloaded: camera_images/camera_01315_20250609_0046.jpg
Waiting for 5 minutes...
Iteration 5 at 2025-06-08 23:51:33.050729
Downloaded: camera_images/camera_01315_20250609_0051.jpg
Zipped images into: camera_images.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>